<details>
<summary><b>Info</b></summary>

**Last Execution:** 2026-07-25

| Package | Version |
|---------|---------|
| **nnsight** | **0.8** |
| Python | 3.12.13 |
| torch | 2.13.0+cu126 |
| transformers | 5.15.0.dev0 |

</details>


# Multiple Token Generation

`model.generate(...)` runs multi-token, autoregressive generation: the model runs one forward pass per new token. The generated **token ids** come back on `tracer.result`. Because interventions in the block run against *every* forward the decode loop makes, nnsight gives you two ways to target particular steps:

- **Bounded** — `tracer.iter[:N]`, a slice, an int, or a list of step indices. You name exactly which steps to touch, so the loop ends on its own and any code after it (like `tracer.result.save()`) still runs.
- **Unbounded** — `tracer.all()` (shorthand for `tracer.iter[:]`), which keeps handing out steps until the model itself stops generating.

Keep that distinction in mind: a bound is what lets the block finish cleanly and keep trailing code, while the unbounded form runs open-endedly and emits a warning (shown below). Prefer a bound whenever you know how many steps you want.

## Setup

In [1]:
import nnsight
from nnsight.modeling.transformers import TransformersModel

model = TransformersModel("openai-community/gpt2", device_map="auto", dispatch=True)

/home/localjadenfk/miniconda3/envs/ndif2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Basic Generation

Use `.generate()` instead of `.trace()` for multi-token generation, and pass `max_new_tokens` to bound it. Read the generated token ids off `tracer.result` — a `[batch, seq]` tensor holding the whole prompt plus the completion.

In [2]:
with model.generate("The Eiffel Tower is in the city of", max_new_tokens=5) as tracer:

    ids = tracer.result.save()

print(ids.shape)
print(model.tokenizer.decode(ids[0]))

torch.Size([1, 15])
The Eiffel Tower is in the city of Paris, and the E


<details class="admonition note">
<summary>Greedy by default</summary>

`generate` goes **through the model**, using the checkpoint's own generation settings, so it is greedy by default — repeat runs give identical ids. Ask for sampling explicitly with `model.generate(..., do_sample=True, top_k=50)`; any keyword is forwarded to the model's `generate`. If you want the task pipeline's decoded records (text, labels) instead of token ids, use `model.pipe(...)`.

</details>

## Bounded Iteration: `tracer.iter[:N]`

To intervene on or collect values from individual steps, loop with `for step in tracer.iter[:N]`. `step` is the real integer step index, and `[:N]` is **bounded** — it collects across the first `N` generated tokens and then stops. Match `N` to `max_new_tokens` so that code after the loop (like saving `tracer.result`) still runs.

To gather per-step values, save a **container** once with `nnsight.save([])` and append the raw values inside the loop — do not call `.save()` on each individual value.

In [3]:
with model.generate("The Eiffel Tower is in the city of", max_new_tokens=5) as tracer:

    tokens = nnsight.save([])

    for step in tracer.iter[:5]:
        tokens.append(model.lm_head.output[0, -1].argmax(dim=-1))

    ids = tracer.result.save()

for i, t in enumerate(tokens):
    print(f"Step {i}: {model.tokenizer.decode(t)}")

print("Full output:", model.tokenizer.decode(ids[0]))

Step 0:  Paris
Step 1: ,
Step 2:  and
Step 3:  the
Step 4:  E
Full output: The Eiffel Tower is in the city of Paris, and the E


## Targeting Specific Steps: slice, int, and list

Subscript `tracer.iter` to target specific steps only — these are all bounded forms:

- a **slice**, `tracer.iter[1:3]`, runs the body for steps 1 and 2 (`stop` is exclusive);
- an **int**, `tracer.iter[0]`, runs just that one step (`0` is the prefill step, which processes the whole prompt);
- a **list**, `tracer.iter[[0, 2, 4]]`, runs only those steps.

Start with the slice form:

In [4]:
with model.generate("The Eiffel Tower is in the city of", max_new_tokens=5) as tracer:

    tokens = nnsight.save([])

    for step in tracer.iter[1:3]:
        tokens.append(model.lm_head.output[0, -1].argmax(dim=-1))

    ids = tracer.result.save()

print(f"Full output: {model.tokenizer.decode(ids[0])}")
print(f"Collected {len(tokens)} tokens from steps 1-2")

Full output: The Eiffel Tower is in the city of Paris, and the E
Collected 2 tokens from steps 1-2


An int targets a single step. `tracer.iter[0]` is the prefill step — the one forward pass that ingests the whole prompt:

In [5]:
with model.generate("The Eiffel Tower is in the city of", max_new_tokens=5) as tracer:

    prefill_pick = nnsight.save([])

    for step in tracer.iter[0]:
        prefill_pick.append(model.lm_head.output[0, -1].argmax(dim=-1))

    ids = tracer.result.save()

print("Step 0 predicts:", model.tokenizer.decode(prefill_pick[0]))

Step 0 predicts:  Paris


A list targets an explicit set of steps — here just steps 0, 2, and 4:

In [6]:
with model.generate("The Eiffel Tower is in the city of", max_new_tokens=5) as tracer:

    picks = nnsight.save([])

    for step in tracer.iter[[0, 2, 4]]:
        picks.append(model.lm_head.output[0, -1].argmax(dim=-1))

    ids = tracer.result.save()

print("Steps 0, 2, 4:", [model.tokenizer.decode(t) for t in picks])

Steps 0, 2, 4: [' Paris', ' and', ' E']


## Conditional Interventions Per Step

Because `step` is a plain integer, an ordinary Python `if` lets you apply different interventions at different points in generation.

In [7]:
with model.generate("The Eiffel Tower is in the city of", max_new_tokens=5) as tracer:

    for step in tracer.iter[:5]:
        if step == 0:
            # Only intervene on the first (prefill) step
            model.transformer.h[0].output[0][:] = 0

    ids = tracer.result.save()

print(model.tokenizer.decode(ids[0]))

The Eiffel Tower is in the city of, but I'm not


## Applying Interventions to Every Step

To run the same intervention on every step, iterate over the full bounded range. Here we zero-ablate layer 0 on every step while collecting the last layer's hidden state each time.

In [8]:
with model.generate("The Eiffel Tower is in the city of", max_new_tokens=5) as tracer:

    hidden_states = nnsight.save([])

    for step in tracer.iter[:5]:
        model.transformer.h[0].output[0][:] = 0
        hidden_states.append(model.transformer.h[-1].output[0])

    ids = tracer.result.save()

print(f"Collected {len(hidden_states)} hidden states")
print(f"Shapes: {[tuple(h.shape) for h in hidden_states]}")
print(f"Output: {model.tokenizer.decode(ids[0])}")

Collected 5 hidden states
Shapes: [(10, 768), (1, 768), (1, 768), (1, 768), (1, 768)]


Output: The Eiffel Tower is in the city of,,,,,


<details class="admonition note">
<summary>Why the first hidden state is larger</summary>

The first step processes the full prompt (10 tokens), so its hidden state has shape `[10, 768]`. Each subsequent step processes a single new token, giving shape `[1, 768]`.

</details>

## Unbounded Iteration: `tracer.all()`

`tracer.all()` is shorthand for the unbounded `tracer.iter[:]`. Instead of naming a step count, it keeps handing out step indices until the model itself stops generating, so it's handy when you don't know the length ahead of time and don't need any code after the loop. Below we capture the warning it emits so we can print it — normally it just prints to stderr.

In [9]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    with model.generate("The Eiffel Tower is in the city of", max_new_tokens=5) as tracer:

        hidden_states = nnsight.save([])

        for step in tracer.all():
            model.transformer.h[0].output[0][:] = 0
            hidden_states.append(model.transformer.h[-1].output[0])

print(f"Collected {len(hidden_states)} hidden states")
for warning in caught:
    print("Warning:", warning.message)

Collected 5 hidden states


The warning reads (paraphrased from the nnsight source) that a location *"was never reached: the model ran fewer iterations than the loop requested. Values from reached iterations are kept."*

**Why it fires:** an open-ended loop can't know in advance when the model will stop, so it always asks for one step *past* the last generated token. That final request lands on a step the model never runs, and the interleaver reports the dangling request as a warning rather than an error. The values from every step that *did* run are still kept — the warning is expected, not a failure. It fires on essentially every `tracer.all()` run, which is the cost of collecting per-step without a bound.

<details class="admonition warning">
<summary>Open-ended iteration drops trailing code — prefer a bound</summary>

`tracer.all()` and `tracer.iter[:]` loop until the model stops generating. That same over-run request that triggers the warning also unwinds the loop **and every line after it**, so a `tracer.result.save()` placed after an open-ended loop never runs. To keep per-step values *and* the final result — and to avoid the warning entirely — use a **bounded** `tracer.iter[:N]` matching `max_new_tokens`, as in the sections above. Reach for `tracer.all()` only when the step count is genuinely unknown.

The old `tracer.next()` manual-stepping API is gone in 0.8 — use these loop forms instead.

</details>

## Calling Generate Directly

Without a `with` block, `generate` just runs and returns the token ids as a tensor.

In [10]:
ids = model.generate("The Eiffel Tower is in the city of", max_new_tokens=3)

print(model.tokenizer.decode(ids[0]))

The Eiffel Tower is in the city of Paris, and
